# DETESTS Dataset Exploration
Investigate comment-level collapsing: how many comment_ids to keep or discard.

In [1]:
import pandas as pd
import re
import os

df = pd.read_csv('data/spanish_subset/train.csv')
detests    = df[df['source'] == 'detests'].copy()
stereohoax = df[df['source'] != 'detests'].copy()

print(f'Total rows         : {len(df):,}')
print(f'DETESTS rows       : {len(detests):,}')
print(f'Stereohoax rows    : {len(stereohoax):,}')
print(f'DETESTS comment_ids: {detests["comment_id"].nunique():,}')

Total rows         : 9,906
DETESTS rows       : 5,629
Stereohoax rows    : 4,277
DETESTS comment_ids: 2,548


## 1. Sentences per comment_id

In [2]:
sentences_per_comment = detests.groupby('comment_id').size()
print('Sentences per comment_id:')
print(sentences_per_comment.describe())
print()
print('Distribution:')
print(sentences_per_comment.value_counts().sort_index().head(20))

Sentences per comment_id:
count    2548.000000
mean        2.209184
std         1.839614
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        24.000000
dtype: float64

Distribution:
1     1187
2      650
3      310
4      167
5      104
6       47
7       36
8       19
9       11
10       5
11       2
12       2
15       2
17       2
18       2
19       1
24       1
Name: count, dtype: int64


## 2. Label consistency per comment_id

In [3]:
label_consistency = detests.groupby('comment_id')['stereotype'].nunique()

consistent_ids   = label_consistency[label_consistency == 1].index
inconsistent_ids = label_consistency[label_consistency > 1].index

print(f'Consistent comment_ids   (keep)   : {len(consistent_ids):,}')
print(f'Inconsistent comment_ids (discard): {len(inconsistent_ids):,}')
print(f'Keep rate: {100 * len(consistent_ids) / label_consistency.shape[0]:.1f}%')
print()

consistent_labels = (
    detests[detests['comment_id'].isin(consistent_ids)]
    .groupby('comment_id')['stereotype']
    .first()
)
print('Label distribution (consistent comment_ids):')
print(consistent_labels.value_counts())

Consistent comment_ids   (keep)   : 2,014
Inconsistent comment_ids (discard): 534
Keep rate: 79.0%

Label distribution (consistent comment_ids):
stereotype
0    1597
1     417
Name: count, dtype: int64


## 3. Collapse consistent comment_ids

In [4]:
def resolve_level(val):
    """Keep inter-comment references, zero out intra-comment sentence refs."""
    if pd.isna(val) or str(val) == '0':
        return '0'
    if re.match(r'^d_\d+_\d+$', str(val)):
        return '0'
    return str(val)

detests_filtered = detests[detests['comment_id'].isin(consistent_ids)].copy()

detests_collapsed = (
    detests_filtered
    .sort_values('id')
    .groupby('comment_id', as_index=False)
    .agg(
        text       = ('text',       ' '.join),
        stereotype = ('stereotype', 'first'),
        source     = ('source',     'first'),
        level2     = ('level2',     'first'),
        level3     = ('level3',     'first'),
        level4     = ('level4',     'first'),
    )
)
detests_collapsed['id'] = detests_collapsed['comment_id']
detests_collapsed['level2'] = detests_collapsed['level2'].apply(resolve_level)
detests_collapsed['level3'] = detests_collapsed['level3'].apply(resolve_level)

print(f'Collapsed DETESTS rows: {len(detests_collapsed):,}')
print(detests_collapsed[['comment_id', 'stereotype', 'level2', 'level3']].head(10))

Collapsed DETESTS rows: 2,014
  comment_id  stereotype  level2  level3
0        d_0           0       0       0
1        d_1           0       0       0
2       d_10           0     d_6   d_144
3      d_100           0       0       0
4     d_1002           1  d_1000  d_1000
5     d_1003           0       0       0
6     d_1004           0  d_1003  d_1003
7     d_1005           0       0       0
8     d_1006           0  d_1005  d_1005
9     d_1007           0  d_1005  d_1005


## 4. How many have external context (level2 / level3)?

In [5]:
has_level2 = (detests_collapsed['level2'] != '0').sum()
has_level3 = (detests_collapsed['level3'] != '0').sum()
has_both   = ((detests_collapsed['level2'] != '0') & (detests_collapsed['level3'] != '0')).sum()

total = len(detests_collapsed)
print(f'Has level2 (parent) : {has_level2:,} / {total:,} ({100*has_level2/total:.1f}%)')
print(f'Has level3 (root)   : {has_level3:,} / {total:,} ({100*has_level3/total:.1f}%)')
print(f'Has both            : {has_both:,} / {total:,} ({100*has_both/total:.1f}%)')
print(f'No context at all   : {total - has_level2:,} / {total:,} ({100*(total-has_level2)/total:.1f}%)')

Has level2 (parent) : 1,265 / 2,014 (62.8%)
Has level3 (root)   : 1,265 / 2,014 (62.8%)
Has both            : 1,265 / 2,014 (62.8%)
No context at all   : 749 / 2,014 (37.2%)


## 5. Merged dataset preview (DETESTS collapsed + Stereohoax)

In [6]:
merged = pd.concat([stereohoax, detests_collapsed], ignore_index=True)
print(f'Stereohoax rows   : {len(stereohoax):,}')
print(f'DETESTS collapsed : {len(detests_collapsed):,}')
print(f'Total merged      : {len(merged):,}')
print()
print('Label distribution in merged dataset:')
print(merged['stereotype'].value_counts())

Stereohoax rows   : 4,277
DETESTS collapsed : 2,014
Total merged      : 6,291

Label distribution in merged dataset:
stereotype
0    4628
1    1663
Name: count, dtype: int64


## 6. Apply same collapse to test.csv and export both

In [7]:
def collapse_detests(df):
    """Filter and collapse DETESTS sentence rows to comment level."""
    detests = df[df['source'] == 'detests'].copy()
    other   = df[df['source'] != 'detests'].copy()

    consistent_ids = (
        detests.groupby('comment_id')['stereotype']
        .nunique()
        .pipe(lambda s: s[s == 1].index)
    )
    print(f'  Consistent comment_ids: {len(consistent_ids):,} / {detests["comment_id"].nunique():,}')

    collapsed = (
        detests[detests['comment_id'].isin(consistent_ids)]
        .sort_values('id')
        .groupby('comment_id', as_index=False)
        .agg(
            text       = ('text',       ' '.join),
            stereotype = ('stereotype', 'first'),
            source     = ('source',     'first'),
            level2     = ('level2',     'first'),
            level3     = ('level3',     'first'),
            level4     = ('level4',     'first'),
        )
    )
    collapsed['id'] = collapsed['comment_id']
    collapsed['level2'] = collapsed['level2'].apply(resolve_level)
    collapsed['level3'] = collapsed['level3'].apply(resolve_level)

    return pd.concat([other, collapsed], ignore_index=True)


df_test = pd.read_csv('data/spanish_subset/test.csv')

print('Processing train.csv...')
train_collapsed = collapse_detests(df)
print(f'  Final train rows: {len(train_collapsed):,}')

print('Processing test.csv...')
test_collapsed = collapse_detests(df_test)
print(f'  Final test rows : {len(test_collapsed):,}')

Processing train.csv...
  Consistent comment_ids: 2,014 / 2,548
  Final train rows: 6,291
Processing test.csv...
  Consistent comment_ids: 376 / 506
  Final test rows : 1,448


In [8]:
out_dir = 'data/spanish_subset_collapsed/'
os.makedirs(out_dir, exist_ok=True)

train_collapsed.to_csv(out_dir + 'train.csv', index=False)
test_collapsed.to_csv(out_dir  + 'test.csv',  index=False)

print(f'Saved to {out_dir}')
print(f'  train.csv : {len(train_collapsed):,} rows')
print(f'  test.csv  : {len(test_collapsed):,} rows')

Saved to data/spanish_subset_collapsed/
  train.csv : 6,291 rows
  test.csv  : 1,448 rows
